# Paper ELMECS - Dataset de Entrenamiento
## Análisis exploratorio del dataset `piuba-bigdata/contextualized_hate_speech` (split train)

**Natalia Debandi**

Dataset: https://huggingface.co/datasets/piuba-bigdata/contextualized_hate_speech

In [2]:
!pip install datasets -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset
import pandas as pd

c:\Users\natal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga del dataset

In [4]:
ds = load_dataset("piuba-bigdata/contextualized_hate_speech")
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'],
        num_rows: 36420
    })
    test: Dataset({
        features: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'],
        num_rows: 11343
    })
    dev: Dataset({
        features: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'],
        num_rows: 9106
    })
})

In [5]:
# Usamos el split de entrenamiento
df_train = ds["train"].to_pandas()
print(f"Shape: {df_train.shape}")
print(f"Columnas: {df_train.columns.tolist()}")
df_train.head(3)

Shape: (36420, 15)
Columnas: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL']


,id,title,text,context_tweet,HATEFUL,body,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL
0,343726,Video: salió de la cárcel por el coronavirus y...,@usuario Uno menos,Video: salió de la cárcel por el coronavirus y...,1,"Un hombre de 46 años, que cumplía una condena ...",0,0,0,0,0,0,0,0,1
1,384799,La muerte de Solange: su padre no pudo darle e...,@usuario #QueSeVayanTodos @usuario @usuario @u...,La carta que escribió Solange antes de morir: ...,0,"Solange Musse, la hija del hombre que había vi...",0,0,0,0,0,0,0,0,0
2,399435,Alberto Fernández negocia la compra de 15 mill...,"@usuario Te falta negociar con Cuba, mamerto.",Alberto Fernández negocia la compra de 15 mill...,0,Alberto Fernández desplegó una estrategia de a...,0,0,0,0,0,0,0,0,0


## 2. Incorporar columna `medio` y `date_tweet`

El dataset de HuggingFace no incluye la columna `medio` ni fecha.  
Se hace un join con `tweets_medios_arg.csv` usando `title` como clave.

El CSV local usa comillas tipográficas (`"..."`) mientras que el dataset HF usa comillas ASCII (`"..."`).  
Se normaliza antes del join para alcanzar ~99% de match.

In [6]:
def normalize_title(s):
    """Normaliza comillas tipográficas y otros caracteres Unicode para el join."""
    if not isinstance(s, str):
        return s
    replacements = {
        '\u201c': '"', '\u201d': '"',   # comillas dobles tipográficas
        '\u2018': "'", '\u2019': "'",   # comillas simples tipográficas
        '\u2013': '-', '\u2014': '-',   # guiones tipográficos
        '\u2026': '...',                # ellipsis
        '\u00a0': ' ',                  # non-breaking space
    }
    for src, dst in replacements.items():
        s = s.replace(src, dst)
    return s.strip()

df_medios = pd.read_csv("data/tweets_medios_arg.csv", usecols=["title", "medio", "date_tweet"])
df_medios_uniq = df_medios.drop_duplicates(subset="title").copy()
df_medios_uniq["title_norm"] = df_medios_uniq["title"].apply(normalize_title)

print(f"Medios únicos en tweets_medios_arg: {df_medios_uniq['medio'].nunique()}")
df_medios_uniq.head(3)

Medios únicos en tweets_medios_arg: 9


,title,medio,date_tweet,title_norm
0,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:03:00.900,Segunda ola de coronavirus: preocupan las reun...
8,La desgarradora confesión de Hugo Maradona: “L...,LANACION,2021-03-30 16:56:05.400,"La desgarradora confesión de Hugo Maradona: ""L..."
15,El dólar blue cada vez más barato: cayó a $ 14...,clarincom,2021-03-30 16:58:00.400,El dólar blue cada vez más barato: cayó a $ 14...


In [7]:
df_train["title_norm"] = df_train["title"].apply(normalize_title)

df = df_train.merge(
    df_medios_uniq[["title_norm", "medio", "date_tweet"]],
    on="title_norm",
    how="left"
)

match_pct = df["medio"].notna().mean()
print(f"Filas con medio asignado: {df['medio'].notna().sum()} ({match_pct:.1%})")
df[["title", "medio", "date_tweet"]].head(3)

Filas con medio asignado: 36620 (99.4%)


,title,medio,date_tweet
0,Video: salió de la cárcel por el coronavirus y...,clarincom,2020-04-22 20:59:00.600
1,La muerte de Solange: su padre no pudo darle e...,infobae,2020-08-21 21:41:03.300
2,Alberto Fernández negocia la compra de 15 mill...,infobae,2020-12-18 09:23:04.800


In [8]:
df["date_tweet"] = pd.to_datetime(df["date_tweet"], errors="coerce")

## 3. Estadísticas por medio

In [9]:
# Cantidad de noticias únicas por medio
noticias_por_medio = (
    df.groupby("medio")["title"]
    .nunique()
    .reset_index()
    .rename(columns={"title": "n_noticias"})
)

# Cantidad total de comentarios por medio
comentarios_por_medio = (
    df.groupby("medio")["text"]
    .count()
    .reset_index()
    .rename(columns={"text": "n_comentarios"})
)

# Fechas primera y última noticia por medio
fechas_por_medio = (
    df.groupby("medio")["date_tweet"]
    .agg(primera_noticia="min", ultima_noticia="max")
    .reset_index()
)

# Unir todo
stats = (
    noticias_por_medio
    .merge(comentarios_por_medio, on="medio")
    .merge(fechas_por_medio, on="medio")
)

# Comentarios promedio por noticia
stats["comentarios_x_noticia"] = (stats["n_comentarios"] / stats["n_noticias"]).round(1)

# Formatear fechas
stats["primera_noticia"] = stats["primera_noticia"].dt.strftime("%Y-%m-%d")
stats["ultima_noticia"]  = stats["ultima_noticia"].dt.strftime("%Y-%m-%d")

stats = stats.sort_values("n_noticias", ascending=False).reset_index(drop=True)

stats

,medio,n_noticias,n_comentarios,primera_noticia,ultima_noticia,comentarios_x_noticia
0,infobae,469,17152,2020-03-01,2021-02-08,36.6
1,clarincom,304,11618,2020-03-06,2021-02-07,38.2
2,LANACION,171,6406,2020-03-02,2021-02-03,37.5
3,cronica,33,988,2020-03-16,2021-01-12,29.9
4,perfilcom,13,456,2020-03-24,2021-02-08,35.1


In [ ]:
print(f"Total de medios: {stats['medio'].nunique()}")
print(f"Total de noticias (train): {stats['n_noticias'].sum()}")
print(f"Total de comentarios (train): {stats['n_comentarios'].sum()}")
print(f"Promedio global de comentarios por noticia: {stats['n_comentarios'].sum() / stats['n_noticias'].sum():.1f}")

In [10]:
df.to_csv("data/contextualized_hate_speech_train.csv", index=False)    